# dEchorate HDF5 RIR Analysis

This notebook provides a robust interface to the dEchorate HDF5 database.
It loads metadata, maps IDs to HDF5 indices, and provides visualization tools for signal analysis (Waveform, Spectrogram, Energy Decay).

In [4]:
# 1. Imports
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import scipy.signal

%matplotlib inline

In [5]:
# 2. Configuration
PROCESSED_METADATA_PATH = Path(
    "../../data/dEchorate/processed/dEchorate_database_cleaned.csv"
)
H5_PATH = Path("../../data/dEchorate/raw/dEchorate_rirs_gzip7.hdf5")
SAMPLING_RATE = 48000
ROOM_DIMS = np.array([6.0, 6.0, 2.4])  # meters

# Validation
assert (
    PROCESSED_METADATA_PATH.exists()
), f"Metadata not found at {PROCESSED_METADATA_PATH}"
assert H5_PATH.exists(), f"HDF5 file not found at {H5_PATH}"

In [6]:
def explore_hdf5_structure(h5_path, max_depth=3):
    """Recursively explore HDF5 file structure to find actual datasets"""

    def print_structure(name, obj, depth=0):
        if depth > max_depth:
            return
        indent = "  " * depth
        if isinstance(obj, h5py.Dataset):
            print(f"{indent}📊 {name}: Dataset {obj.shape} {obj.dtype}")
        elif isinstance(obj, h5py.Group):
            print(f"{indent}📁 {name}: Group")

    with h5py.File(h5_path, "r") as f:
        print(f"HDF5 File: {h5_path.name}")
        f.visititems(print_structure)


explore_hdf5_structure(H5_PATH)

HDF5 File: dEchorate_rirs_gzip7.hdf5
📁 rir: Group


RuntimeError: Object visitation failed (addr overflow, addr = 4024414617, size = 176, eoa = 2048)

In [ ]:
# 3. Load Metadata & ID Mapping
print("Loading metadata...")
df = pd.read_csv(PROCESSED_METADATA_PATH)

# Ensure IDs are integers for consistent mapping
for col in ["src_id", "mic_id"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(-1).astype(int)

# Build room_code from reflection columns
# room_code format: 'fcwsen' where each digit is 0 or 1
df['room_code'] = (
    df['room_rfl_floor'].astype(int).astype(str) +
    df['room_rfl_ceiling'].astype(int).astype(str) +
    df['room_rfl_west'].astype(int).astype(str) +
    df['room_rfl_south'].astype(int).astype(str) +
    df['room_rfl_east'].astype(int).astype(str) +
    df['room_rfl_north'].astype(int).astype(str)
)

room_list = sorted(df['room_code'].unique())
src_list = sorted(df['src_id'].unique())
mic_list = sorted(df['mic_id'].unique())

print(f"Found {len(room_list)} unique room configurations")
print(f"Unique room codes: {room_list[:5]}...")
print(f"Found {len(src_list)} Sources: {src_list}")
print(f"Found {len(mic_list)} Mics: {mic_list}")


Loading metadata...
Found 2 Rooms: [np.int64(0), np.int64(1)]
Found 3 Sources: [np.int64(0), np.int64(1), np.int64(2)]
Found 30 Mics: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29)]


In [ ]:
# 4. Define Data Loader
class DechorateLoader:
    def __init__(self, h5_path, metadata_df):
        self.h5_path = h5_path
        self.metadata_df = metadata_df
        
        if not h5_path.exists():
            raise FileNotFoundError(f"HDF5 file not found: {h5_path}")
        
        with h5py.File(h5_path, 'r') as f:
            if 'rir' not in f:
                raise ValueError("HDF5 file missing 'rir' group")
        
        print(f"✓ Connected to {h5_path.name}")
        print(f"✓ Loaded metadata with {len(metadata_df)} entries")
    
    def get_rir(self, room_code, src_id, mic_id, recording_offset=2000):
        """
        Retrieve RIR for given configuration.
        """
        # Construct hierarchical path
        group_path = f'rir/{room_code}/rir/{src_id}/{mic_id}'
        
        try:
            with h5py.File(self.h5_path, 'r') as f:
                rir = f[group_path][:]
                rir = rir[recording_offset:]
            return rir.squeeze()
        
        except KeyError:
            raise ValueError(
                f"RIR not found for room={room_code}, src={src_id}, mic={mic_id}. "
                f"Check metadata CSV for valid combinations."
            )

loader = DechorateLoader(H5_PATH, df)


AttributeError: 'Group' object has no attribute 'shape'

In [ ]:
# 5. Signal Processing & Visualization Tools
def compute_edc(rir):
    """Compute Energy Decay Curve using Schroeder Integration."""
    # Standard backward integration
    energy = rir**2
    edc = np.cumsum(energy[::-1])[::-1]
    # Normalize to dB
    return 10 * np.log10(edc / (edc.max() + 1e-12) + 1e-12)


def plot_rir_analysis(rir, fs=SAMPLING_RATE):
    t = np.arange(len(rir)) / fs

    fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=False)

    # 1. Waveform
    axes[0].plot(t, rir, color="tab:blue", lw=0.5)
    axes[0].set_title("Impulse Response (Time Domain)")
    axes[0].set_ylabel("Amplitude")
    axes[0].set_xlim(0, t.max())
    axes[0].grid(True, alpha=0.3)

    # 2. Spectrogram
    f, t_spec, Sxx = scipy.signal.spectrogram(rir, fs, nperseg=256, noverlap=128)
    pcm = axes[1].pcolormesh(
        t_spec, f, 10 * np.log10(Sxx + 1e-12), shading="gouraud", cmap="inferno"
    )
    axes[1].set_title("Spectrogram")
    axes[1].set_ylabel("Frequency (Hz)")
    axes[1].set_xlim(0, t.max())
    fig.colorbar(pcm, ax=axes[1], label="Power (dB)")

    # 3. Energy Decay Curve
    edc = compute_edc(rir)
    axes[2].plot(t, edc, color="tab:red", lw=1.5)
    axes[2].set_title("Energy Decay Curve (Schroeder Integration)")
    axes[2].set_xlabel("Time (s)")
    axes[2].set_ylabel("Energy (dB)")
    axes[2].set_xlim(0, t.max())
    axes[2].set_ylim(-60, 0)  # Standard T60 range
    axes[2].grid(True, which="both", alpha=0.3)

    plt.tight_layout()
    plt.show()

## 6. Explore Data
Select a Room, Source, and Mic to visualize.

In [ ]:
# Example Selection using available metadata
try:
    # Pick a row that has a chirp signal
    test_row = df[df['src_signal'] == 'chirp'].iloc[0]
    
    target_room_code = test_row['room_code']
    target_src = int(test_row['src_id'])
    target_mic = int(test_row['mic_id'])
    
    print(f"Fetching RIR for Room={target_room_code}, Src={target_src}, Mic={target_mic}...")
    rir_signal = loader.get_rir(target_room_code, target_src, target_mic)
    plot_rir_analysis(rir_signal)
except Exception as e:
    print(f"Error: {e}")


In [ ]:
# 7. Bulk Statistics (Optional)
# Quickly check max amplitude across a slice to check for clipping or silence
print("Checking RIR statistics for first source in Room 0...")
with h5py.File(H5_PATH, "r") as f:
    # Slice: Room 0, Src 0, All Mics, All Time
    # Indices: [room_idx, src_idx, mic_idx, time]
    # Let's assume indices map 0->0 for simplicity here, or use maps
    slice_data = f["rir"][0, 0, :, :]

    print(f"Slice shape: {slice_data.shape}")
    print(f"Max Amp: {np.max(np.abs(slice_data)):.4f}")
    print(f"Mean Power: {np.mean(slice_data**2):.6f}")